In [1]:
import pandas as pd
import json
import os
import glob
import re

# =============================================================================
# Параметры
# =============================================================================
INPUT_DIR = "../data/output"
DICT_PATH = "../data/dictionary/result/dictionary_en_ru.json"
OUTPUT_DIR = "../data/output"
FILE_PATTERN = "guns_result_*.csv"

# =============================================================================
# 1. Поиск самого свежего файла пушек
# =============================================================================
def get_latest_guns_file(directory, pattern):
    """
    Возвращает путь к самому новому файлу guns_result_*.csv по дате в имени.
    Если файлов нет, райзит исключение.
    """
    search_path = os.path.join(directory, pattern)
    files = glob.glob(search_path)
    if not files:
        raise FileNotFoundError(f"Не найдено файлов, соответствующих шаблону {search_path}")
    # Сортировка по имени даст правильный хронологический порядок,
    # т.к. дата записана в формате YYYYMMDD_HHMMSS
    files.sort()
    latest = files[-1]
    print(f"Выбран файл: {os.path.basename(latest)}")
    return latest

# =============================================================================
# 2. Загрузка словаря переводов
# =============================================================================
def load_translation_dict(json_path):
    """
    Загружает JSON-словарь и строит обратный индекс:
    lowercase(en) -> set уникальных непустых переводов (ru).
    Возвращает dict.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    
    lookup = {}
    for entry in raw.values():
        en_text = entry.get("en", "")
        ru_text = entry.get("ru", "")
        if not en_text:
            continue
        key = en_text.strip().lower()
        # Игнорируем переводы, состоящие только из пробелов
        if ru_text.strip():
            lookup.setdefault(key, set()).add(ru_text.strip())
        else:
            # Если перевода нет (пустая строка), сохраняем, но без текста
            # но чтобы отличать "не найден" от "пустой перевод", 
            # не будем добавлять пустые строки в set.
            # Если запись есть, но ru пустой, перевода по сути нет.
            pass
    return lookup

# =============================================================================
# 3. Функция перевода одной строки
# =============================================================================
def translate_text(text, lookup_dict):
    """
    Ищет перевод для text через lookup_dict (нижний регистр).
    Возвращает ru строку или специальные пометки.
    """
    if not isinstance(text, str) or not text.strip():
        return ""  # если исходное поле пустое, оставляем пустым
    
    query = text.strip().lower()
    candidates = lookup_dict.get(query, set())
    
    if not candidates:
        return "(перевод не найден)"
    elif len(candidates) > 1:
        return "(требуется ручная проверка)"
    else:
        return next(iter(candidates))

# =============================================================================
# 4. Перевод столбца Drop Source (значения через запятую)
# =============================================================================
def translate_drop_source(drop_source_text, lookup_dict):
    """
    Разделяет строку по запятым, переводит каждый элемент,
    собирает результат обратно через ', '.
    """
    if not isinstance(drop_source_text, str) or not drop_source_text.strip():
        return ""
    
    parts = [p.strip() for p in drop_source_text.split(",") if p.strip()]
    translated_parts = [translate_text(p, lookup_dict) for p in parts]
    return ", ".join(translated_parts)

# =============================================================================
# Основной блок выполнения
# =============================================================================
try:
    # Поиск файла
    latest_file = get_latest_guns_file(INPUT_DIR, FILE_PATTERN)
    
    # Загрузка переводов
    print("Загрузка словаря...")
    en_ru_lookup = load_translation_dict(DICT_PATH)
    print(f"Загружено {len(en_ru_lookup)} уникальных английских фраз.")
    
    # Чтение CSV
    df = pd.read_csv(latest_file)
    print(f"Прочитано строк: {len(df)}")
    
    # Проверяем наличие обязательных колонок
    required_cols = ["Name", "Red Text", "Drop Source"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"В файле отсутствуют необходимые столбцы: {missing}")
    
    # Перевод столбцов
    print("Выполняю перевод...")
    df["Name_ru"] = df["Name"].apply(lambda x: translate_text(x, en_ru_lookup))
    df["Red Text_ru"] = df["Red Text"].apply(lambda x: translate_text(x, en_ru_lookup))
    df["Drop Source_ru"] = df["Drop Source"].apply(lambda x: translate_drop_source(x, en_ru_lookup))
    
    # Сохранение результата
    base_name = os.path.basename(latest_file)
    name_without_ext, ext = os.path.splitext(base_name)
    output_name = f"{name_without_ext}_with_ru.csv"
    output_path = os.path.join(OUTPUT_DIR, output_name)
    
    df.to_csv(output_path, index=False, encoding='utf-8-sig')  # UTF-8 с BOM для Excel
    print(f"Результат сохранён: {output_path}")
    
except Exception as e:
    print(f"Ошибка: {e}")

Выбран файл: guns_result_20260806_193323.csv
Загрузка словаря...
Загружено 61228 уникальных английских фраз.
Прочитано строк: 133
Выполняю перевод...
Результат сохранён: ../data/output/guns_result_20260806_193323_with_ru.csv
